Computed in four steps:

1. **Split the sample into thirds by time.** Years with enough observations are collected and cut into three equal blocks — early, mid, late — thirds of the *timeline*, not of the value range.
2. **Take the median of the values in each period.** The median of every observation falling in those years (not the median of annual medians), so a single crisis observation can't drag a period's centre.
3. **Divide by a robust scale**, `sigma_f = 1.4826 × median(|x − median(x)|)` — the scaled MAD over the *whole* sample. This normalizes across factors of wildly different units (e.g. `cpi_shelter` moves ~96 index points, `csize_to_shrout` is a ratio near 1e-5), and using MAD rather than `std()` means a single crisis observation can't inflate the denominator and mask genuine drift.
4. **Take max minus min across the three periods — not late minus early.** This is the load-bearing design choice. A series going 100 → 1,800 → 20 gives `late − early = −80` (looks flat) but `max − min = 1,780` (correctly enormous). This is exactly the shape of `tga`, which an earlier monotone-trend test missed entirely and which was only caught by eye.

Reading: "the centre of the distribution moved *X* robust standard deviations across the sample."

### Why 1.80 is the reference point
A pure, noiseless linear trend from 0 to 1 produces period medians at 1/6, 1/2, 5/6, so the numerator is 2/3; a uniform [0,1] distribution has MAD 0.25 so `sigma_f = 0.371`; giving `shift_max = 0.667 / 0.371 = 1.80`. So **1.80 is what a perfectly clean straight line scores.** Below it, noise is diluting the drift; above it, the series is more step-like or convex than a straight line. A threshold just above 1.80 means "the drift is at least as clean as a straight line."

### Why not `drift_ratio = std(annual means) / mean(annual stds)` (the previous statistic)
That statistic's denominator measures *within-year* variability, which fails in both directions:
- `FirmAge` scored 66.24 (highest in the dataset) purely because it barely moves within a year, making the denominator tiny — but its `shift_max` is only 1.85 (modest drift). The old ratio was measuring smoothness, not drift.
- `csize_to_shrout` scored 1.02 (below the 1.71 median, would not have been flagged) because trade size is noisy day-to-day, inflating the denominator — but its `shift_max` is 2.10, and a separate boundary diagnostic had already confirmed it non-stationary.

Across 1,505 candidates, `drift_ratio` had a median of 1.71 with 76% above 1.0 — no threshold separated anything. `shift_max` normalizes by overall spread instead, asking the question an expanding z-score actually cares about: did the centre move relative to how much this thing varies at all.

## Two Stated Limitations
- **A step and a ramp score the same.** A series flat at 10 for 14 years then flat at 50 for 7 produces the same `shift_max` as a smooth climb. The three period medians are printed alongside the score, and a `shape` label (rising/falling/humped/dipped) is derived from them so the shapes can be told apart by eye.
- **A V that returns to its origin can be missed.** Down then back up to the starting level gives `med_early ≈ med_late`; if `med_mid` isn't extreme enough, the score stays low. Rare in practice — using more than three periods would catch it but would shift the 1.80 benchmark, so three periods are kept.

## What Is Measured, and What Counts as a Candidate
Every column in every table is measured — nothing is skipped. The report then segments columns into three groups, because a high score means something different in each:

- **`candidate`** — genuinely eligible for a transform (raw levels, cwmean/cwstd/spread moments).
- **`differenced`** — already a change series (`_diff`, `_chg`, `_mom`, `_yoy`, `_ret`, `_accel`, `_cum_*`, `_rel_*`, `_vs_ma*`, and named exceptions like `dlyret`, `mktrf`, `b10ret`, world-index returns `widx_*`). A high score here means the *volatility of the changes* is drifting, not the level — differencing again would be wrong. `cpi_core_yoy` scoring high, for example, reflects inflation genuinely regime-shifting, which is signal, not an artefact.
- **`scale_invariant`** — `_cwskew` and `_cwkurt`, computed on standardized deviations `(x − mean)/std`, so any rescaling of `x` cancels out mathematically. These are used as a sanity check on the statistic itself: since they mathematically cannot inherit a level drift, they should score lower than `candidate`. If they didn't, that would suggest `shift_max` is measuring something every column shares rather than genuine drift.

Column-to-moment mapping (`split_moment`) requires that a `_cwmean` sibling actually exists before treating a `_cwstd`/`_cwskew`/`_cwkurt`/`_spread` suffix as a moment suffix — otherwise macro series that merely happen to end in "spread" (`other_spread`, `bull_bear_spread`, `vix_term_spread`) would be misclassified.

## Method Details
- **Per-column scan (`scan`)**: requires ≥30 valid observations and ≥5 years with ≥10 observations each; splits into year-thirds, computes the three period medians, the whole-sample MAD-based scale, and `shift_max`.
- **Differenced detection (`is_differenced`)**: token match on underscore-separated name parts (so `retail` does not spuriously match `ret`) against a set of change-related tokens, plus an explicit exact-name list and a `widx_` prefix rule for world-index return columns.

## Report Sections
- **Coverage** — column and base-factor counts per group (candidate / differenced / scale_invariant), plus count of skipped columns.
- **Shift_max distribution** — a banded histogram (0–0.5, 0.5–1.0, …, >4.0) shown side-by-side across all three groups, since the comparison across groups is itself a validity check on the statistic: scale-invariant should sit lowest (mathematically incapable of drift), differenced should sit low (no level to drift), and candidate is the only group where a high score implies action.
- **Cumulative threshold table** — share of each group at or above several candidate thresholds (1.5–4.0).
- **Percentiles** per group.
- **Validating comparison** — candidate median vs. scale-invariant median, with an explicit pass/fail message: if scale-invariant is *not* lower, that flags either genuine cross-sectional shape drift or a `split_moment` mislabelling bug.
- **Threshold table (columns and base factors)** — base-factor counts matter more than column counts for workload, since one drop/transform decision covers all of a factor's moments and both tables it appears in.
- **Shape breakdown** of high-scoring candidates (rising/falling/humped/dipped), specifically to surface `tga`-like reversal cases a monotone test would miss.
- **Top 40 base factors by shift_max**, with their three period medians printed for interpretation.
- **By-table breakdown** of candidate median and count above 1.8 / 2.5.
- **Already-differenced-but-still-drifting** factors at `shift_max ≥ 2.0** — flags where the volatility of changes is drifting (e.g. `cpi_core_yoy`, `case_shiller_yoy` — real regime shifts, not artefacts) as opposed to something needing correction.
- **Scale-invariant sanity check** — explicit median and count-above-threshold for cwskew/cwkurt.
- **Targeted checks** — specific factors carried over from an earlier boundary diagnostic (`csize_to_shrout`, `oi_wt_theta`, `tga_diff`, etc.).
- **CFTC raw counts vs. OI-normalized** — directly compares raw position counts (which grew with total open interest ~10x over the sample) against their `_pct` normalized counterparts, to distinguish "these are redundant with a stationary version" (a drop) from "these need a transform."

## Output
- `Data/Data_Collection/Final/Stage_3_Cleaning/stationarity_scan.csv` — one row per column across all five input tables, with `shift_max`, shape, three period medians, robust scale, observation/year counts, differenced/scale-invariant/candidate group label.

No data is modified. The next step (referenced in code as "03b") is to read this scan, set a threshold, and decide drops/transforms by hand — though per the Stage 3 notebook 01 docstring, that transform notebook was ultimately abandoned in favor of handling everything the scan surfaced as redundancy via explicit drops.

In [4]:
"""
Stage 3 Cleaning — 03a: Stationarity Scan
=========================================
SCAN ONLY. Measures every factor and reports. Applies no transforms.

WHY THIS EXISTS
---------------
Non-stationarity is the one problem z-scoring cannot fix. An expanding mean lags
a drifting level by construction, so the z-score grows without bound and then
pins at the clip boundary. No robust scale estimator, no cap, no floor helps --
the MEAN is what is wrong, not the variance.

Its signature in the z-scores is ambiguous: a trending feature and a heavy-tailed
feature both show high boundary mass. Measured on raw values it is unambiguous.


THE STATISTIC
-------------
    shift_max = (max - min of period medians) / sigma_f

Computed in four steps.

  1. SPLIT THE SAMPLE INTO THIRDS BY TIME.
     Years with enough observations are collected and cut into three equal
     blocks: early, mid, late. Thirds of the TIMELINE, not of the value range.

  2. TAKE THE MEDIAN OF THE VALUES IN EACH PERIOD.
     The median of every observation falling in those years -- not the median of
     annual medians. Medians rather than means so that one crisis observation
     cannot drag a period's centre.

  3. DIVIDE BY A ROBUST SCALE.
         sigma_f = 1.4826 * median(|x - median(x)|)
     the scaled MAD over the WHOLE sample. Medians alone are not comparable
     across factors -- cpi_shelter moved 96 index points, csize_to_shrout is a
     ratio near 1e-5. Dividing by sigma_f puts both in sigma units. The 1.4826
     rescales MAD to equal a standard deviation for normal data, so the number
     reads in familiar units. A median-based scale is used rather than std() so
     that one crisis observation cannot inflate the denominator and hide a
     genuine drift.

  4. TAKE MAX MINUS MIN ACROSS THE THREE PERIODS.
     Not late minus early. This is the design decision that matters. Consider a
     series going 100 -> 1,800 -> 20:
         late - early = -80      looks almost flat
         max  - min   = 1,780    correctly enormous
     That is tga. The earlier boundary diagnostic classified trends with a
     MONOTONE test, missed tga entirely, and it was only caught by eye. Max
     minus min catches any shape.

Reading: "the centre of the distribution moved X robust standard deviations
across the sample."


WHY 1.80 IS THE REFERENCE POINT
-------------------------------
Take a PURE NOISELESS LINEAR TREND running from 0 to 1 over the sample.

    Numerator:   the three period medians land at the midpoints of each third,
                 i.e. 1/6, 1/2, 5/6.  So max - min = 5/6 - 1/6 = 2/3.

    Denominator: a uniform distribution on [0,1] has median 0.5, and half the
                 points lie within 0.25 of it, so MAD = 0.25 and
                 sigma_f = 1.4826 * 0.25 = 0.371.

    shift_max  = 0.667 / 0.371 = 1.80

So 1.80 is what a perfectly clean straight line produces. Below it, noise is
diluting the drift. Above it, the series is even more step-like or convex than a
straight line. A threshold just above 1.80 therefore means "the drift is at
least as clean as a straight line".


WHY NOT drift_ratio = std(annual means) / mean(annual stds)
-----------------------------------------------------------
That was the previous version's headline statistic and it failed in both
directions, because its denominator measures WITHIN-YEAR variability rather than
overall spread:

  FirmAge scored 66.24 -- the highest in the dataset. Firm age barely moves
  within a year, so the denominator was tiny and the ratio exploded. Its
  shift_max is 1.85: the centre moved modestly. The ratio was measuring
  SMOOTHNESS, not drift.

  csize_to_shrout scored 1.02, below the median of 1.71, and would not have been
  flagged. Trade size is noisy day to day, so the denominator was large and the
  ratio was suppressed. Its shift_max is 2.10, and the boundary diagnostic had
  already confirmed it as non-stationary.

Across 1,505 candidates drift_ratio had a median of 1.71 and 76% above 1.0 --
no threshold separated anything. shift_max normalises by the OVERALL spread, so
it asks "did the centre move relative to how much this thing varies at all",
which is the question an expanding z-score actually cares about.


TWO LIMITATIONS, STATED RATHER THAN HIDDEN
------------------------------------------
  A step and a ramp score the same. A series flat at 10 for fourteen years then
  flat at 50 for seven produces the same shift_max as a smooth climb. The three
  period medians are printed so the shapes can be told apart by eye, and a shape
  label (rising / falling / humped / dipped) is derived from them.

  A V that returns to its origin can be missed. Down then back up to the
  starting level gives med_early ~ med_late, and if med_mid is not extreme
  enough the score stays low. Rare in practice. Using more than three periods
  would catch it but would move the 1.80 benchmark, so three is kept.


WHAT IS MEASURED, AND WHAT IS A CANDIDATE
-----------------------------------------
EVERY column is measured. Nothing is skipped. The REPORT then segments into
three groups, because a high score means something different in each:

  candidate         eligible for a transform.
  already differenced   _diff, _chg, _mom, _yoy, _ret, _accel, _cum_*, _rel_*,
                    _vs_ma*, and the raw return series. A high score here means
                    the VOLATILITY OF THE CHANGES is drifting, not the level --
                    differencing again would be wrong. cpi_core_yoy scoring high
                    is inflation genuinely regime-shifting, which is signal.
  scale-invariant   _cwskew and _cwkurt, computed on standardised deviations
                    (x - mean)/std, so any rescaling of x cancels. A secular
                    decline in trade size mathematically cannot touch them. Only
                    _cwmean, _cwstd and _spread inherit a trend -- and a
                    transform must hit all three of a family or the moments
                    diverge.

Input:   Stage_3_Cleaning/{4 aggregate tables}.parquet + weekly_raw.parquet
Output:  Stage_3_Cleaning/stationarity_scan.csv
"""

import numpy as np
import pandas as pd
from pathlib import Path

DIR = Path('../../../Data/Data_Collection/Final/Stage_3_Cleaning')

TABLES = {
    'agg_daily_means':          'agg_market_daily_means.parquet',
    'agg_daily_full_moments':   'agg_market_daily_full_moments.parquet',
    'agg_monthly_means':        'agg_market_monthly_means.parquet',
    'agg_monthly_full_moments': 'agg_market_monthly_full_moments.parquet',
    'weekly':                   'weekly_raw.parquet',
}
SHORT = {'agg_daily_means': 'dm', 'agg_daily_full_moments': 'dfm',
         'agg_monthly_means': 'mm', 'agg_monthly_full_moments': 'mfm',
         'weekly': 'wk'}

META = {'date', 'target_daily_return', 'target_monthly_return'}
MOMENTS = ('_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread')

PURE_TREND = 1.80          # what a noiseless straight line scores
THRESHOLDS = [1.5, 1.8, 2.0, 2.5, 3.0, 4.0]

# ── Already-differenced detection ───────────────────────────────────────────
# Token match on underscore-separated parts, so 'retail' does NOT match 'ret'
# (splitting gives ['retail'], never ['ret', 'ail']).
DIFF_TOKENS = {'diff', 'chg', 'mom', 'yoy', 'accel', 'ret', 'cum', 'rel',
               'vs', 'revision', 'surprise'}

# Return and change series whose names carry no such token.
DIFF_EXACT = {
    'dlyret', 'dlyretx', 'dlyreti', 'ret_mkt_m', 'open_to_close_ret',
    'intraday_drift', 'midday_drift', 'morning_drift',
    'mktrf', 'smb', 'hml', 'rmw', 'cma', 'umd',
    'b30ret', 'b20ret', 'b10ret', 'b7ret', 'b5ret', 'b2ret', 'b1ret',
    't90ret', 't30ret', 'cpiret',
    'lev_net_chg', 'am_net_chg',
}
DIFF_PREFIX = ('widx_',)


def is_differenced(base: str) -> bool:
    if base in DIFF_EXACT or base.startswith(DIFF_PREFIX):
        return True
    return bool(set(base.split('_')) & DIFF_TOKENS)


def split_moment(col: str, colset: set) -> tuple:
    """(base, moment). A suffix counts only if base_cwmean also exists --
    otherwise macro series merely ENDING in the word 'spread' (other_spread,
    bull_bear_spread, vix_term_spread) are misread as spread moments."""
    for s in MOMENTS:
        if col.endswith(s):
            cand = col[: -len(s)]
            if f'{cand}_cwmean' in colset:
                return cand, s[1:]
            break
    return col, 'level'


def shape_of(e: float, m: float, l: float) -> str:
    """Read the drift's shape from the three period medians."""
    if e < m < l:
        return 'rising'
    if e > m > l:
        return 'falling'
    if m > e and m > l:
        return 'humped'
    if m < e and m < l:
        return 'dipped'
    return 'flat'


def scan(s: pd.Series, years: pd.Series, min_obs_year=10, min_years=5):
    ok = s.notna().to_numpy()
    if ok.sum() < 30:
        return None

    v, y = s[ok], years[ok]

    # Only years with enough observations. Excludes partial first/last years and
    # any year a late-starting factor barely covers.
    counts = v.groupby(y).count()
    good = counts[counts >= min_obs_year].index
    if len(good) < min_years:
        return None
    v, y = v[y.isin(good)], y[y.isin(good)]

    # ── Step 1: thirds of the TIMELINE ──────────────────────────────────────
    yrs = sorted(good)
    k = max(len(yrs) // 3, 1)
    early, mid, late = yrs[:k], (yrs[k:-k] or yrs[k:k + 1]), yrs[-k:]

    # ── Step 2: median of the VALUES in each period ─────────────────────────
    e = float(v[y.isin(early)].median())
    m = float(v[y.isin(mid)].median())
    l = float(v[y.isin(late)].median())

    # ── Step 3: robust scale over the whole sample ──────────────────────────
    arr = v.to_numpy()
    sf = float(1.4826 * np.median(np.abs(arr - np.median(arr))))

    # ── Step 4: max minus min, NOT late minus early ─────────────────────────
    shift = (max(e, m, l) - min(e, m, l)) / sf if sf > 1e-12 else np.nan

    return {
        'shift_max': float(shift),
        'shape': shape_of(e, m, l),
        'med_early': e, 'med_mid': m, 'med_late': l,
        'sigma_f': sf,
        'n_obs': int(ok.sum()), 'n_years': len(yrs),
        'first_year': int(yrs[0]), 'last_year': int(yrs[-1]),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# RUN — every column in every table
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 78)
print("STAGE 3 CLEANING — 03a: STATIONARITY SCAN")
print("=" * 78)
print(f"\n  shift_max = (max - min of period medians) / sigma_f")
print(f"  A pure noiseless linear trend scores {PURE_TREND:.2f}")

rows, skipped = [], 0
for tname, fname in TABLES.items():
    path = DIR / fname
    if not path.exists():
        print(f"\n  MISSING: {path}")
        continue

    df = pd.read_parquet(path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    years = df['date'].dt.year

    feats = [c for c in df.columns if c not in META]
    colset = set(feats)

    for c in feats:
        base, moment = split_moment(c, colset)
        r = scan(df[c], years)
        if r is None:
            skipped += 1
            continue
        rows.append({'table': tname, 'column': c, 'base': base,
                     'moment': moment,
                     'differenced': is_differenced(base),
                     'scale_invariant': moment in ('cwskew', 'cwkurt'),
                     **r})

    print(f"  {tname:<26} {len(df):>6,} rows x {len(feats):>5} features")

sc = pd.DataFrame(rows)
sc['group'] = np.where(sc['scale_invariant'], 'scale_invariant',
              np.where(sc['differenced'], 'differenced', 'candidate'))
sc.to_csv(DIR / 'stationarity_scan.csv', index=False)

# ═══════════════════════════════════════════════════════════════════════════════
# REPORT
# ═══════════════════════════════════════════════════════════════════════════════

cand = sc[sc['group'] == 'candidate']

print(f"\n  COVERAGE")
print(f"    Every column is measured. The report segments them because a high")
print(f"    score means something different in each group.")
print(f"\n    {'group':<18} {'columns':>8} {'base factors':>13}")
print(f"    {'-' * 41}")
for g in ('candidate', 'differenced', 'scale_invariant'):
    sub = sc[sc['group'] == g]
    print(f"    {g:<18} {len(sub):>8,} {sub['base'].nunique():>13,}")
print(f"    {'-' * 41}")
print(f"    {'total':<18} {len(sc):>8,} {sc['base'].nunique():>13,}")
if skipped:
    print(f"    {skipped} columns skipped (fewer than 30 obs or 5 usable years)")

# ── Distribution, all three groups side by side ─────────────────────────────
# Shown together because the comparison is itself a check on the statistic:
#
#   scale_invariant SHOULD sit lowest. cwskew and cwkurt divide out any
#   rescaling of x, so they are mathematically incapable of inheriting a level
#   drift. If their distribution looked like the candidates', shift_max would be
#   measuring something every column shares rather than level drift.
#
#   differenced should also sit low, since a change series has no level to
#   drift. Where it does not, the volatility of the changes is drifting --
#   real, but not fixable by differencing again.
#
#   candidate is the only group where a high score implies a transform.
GROUPS = ('candidate', 'differenced', 'scale_invariant')

bands = [(0, 0.5, 'centre barely moves'), (0.5, 1.0, ''), (1.0, 1.5, ''),
         (1.5, 1.8, 'approaching a trend'), (1.8, 2.5, 'trend-like'),
         (2.5, 4.0, 'strong'), (4.0, np.inf, 'severe')]

print(f"\n  SHIFT_MAX DISTRIBUTION — ALL THREE GROUPS")
print(f"    Percentages are within-group, so the columns are comparable even")
print(f"    though the groups differ in size.")
print(f"\n    {'band':<14} "
      f"{'candidate':>18} {'differenced':>18} {'scale_inv':>18}   note")
print(f"    {'':14} "
      f"{'n':>8}{'%':>10} {'n':>8}{'%':>10} {'n':>8}{'%':>10}")
print(f"    {'-' * 88}")

for lo, hi, note in bands:
    rng = f"{lo:.1f} - {hi:.1f}" if np.isfinite(hi) else f">{lo:.1f}"
    line = f"    {rng:<14} "
    for g in GROUPS:
        sub = sc[sc['group'] == g]
        n = int(((sub['shift_max'] >= lo) & (sub['shift_max'] < hi)).sum())
        pct = n / len(sub) * 100 if len(sub) else 0.0
        line += f"{n:>8}{pct:>9.1f}% "
    print(line + f"  {note}")

print(f"    {'-' * 88}")
line = f"    {'total':<14} "
for g in GROUPS:
    sub = sc[sc['group'] == g]
    line += f"{len(sub):>8}{100.0:>9.1f}% "
print(line)

# Cumulative share at or above each band edge -- reads as "what fraction of this
# group would be caught by a threshold here".
print(f"\n  CUMULATIVE: share of each group AT OR ABOVE a threshold")
print(f"    {'threshold':<12} {'candidate':>20} {'differenced':>20} "
      f"{'scale_inv':>20}")
print(f"    {'-' * 74}")
for t in THRESHOLDS:
    line = f"    {t:<12.1f} "
    for g in GROUPS:
        sub = sc[sc['group'] == g]
        n = int((sub['shift_max'] >= t).sum())
        pct = n / len(sub) * 100 if len(sub) else 0.0
        line += f"{n:>10} ({pct:>5.1f}%) "
    print(line)

print(f"\n  PERCENTILES")
print(f"    {'group':<18} {'min':>7} {'p25':>7} {'p50':>7} {'p75':>7} "
      f"{'p90':>7} {'p95':>7} {'p99':>7} {'max':>8}")
print(f"    {'-' * 76}")
for g in GROUPS:
    s = sc[sc['group'] == g]['shift_max']
    print(f"    {g:<18} {s.min():>7.2f} "
          + " ".join(f"{s.quantile(q):>7.2f}"
                     for q in (0.25, 0.50, 0.75, 0.90, 0.95, 0.99))
          + f" {s.max():>8.2f}")

# The comparison that validates the statistic
med_c = sc[sc['group'] == 'candidate']['shift_max'].median()
med_s = sc[sc['group'] == 'scale_invariant']['shift_max'].median()
print(f"\n    Candidates median {med_c:.2f} vs scale-invariant {med_s:.2f}.")
if med_s < med_c:
    print(f"    Scale-invariant sits lower, as it must -- confirming shift_max")
    print(f"    tracks level drift rather than something every column shares.")
else:
    print(f"    ** Scale-invariant is NOT lower. cwskew/cwkurt cannot inherit a")
    print(f"    level drift, so either the cross-sectional SHAPE is genuinely")
    print(f"    changing over time, or split_moment has mislabelled columns. **")

# ── Threshold table: columns AND base factors ───────────────────────────────
# Base factors are what matter for workload -- one decision covers all its
# moments and both tables it appears in.
print(f"\n  IF THE THRESHOLD WERE SET AT...")
print(f"    {'thresh':>7} {'columns':>9} {'% of cand':>10} "
      f"{'base factors':>13}  note")
print(f"    {'-' * 58}")
for t in THRESHOLDS:
    sub = cand[cand['shift_max'] >= t]
    note = 'pure linear trend' if abs(t - PURE_TREND) < 0.01 else ''
    print(f"    {t:>7.1f} {len(sub):>9,} {len(sub) / len(cand) * 100:>9.1f}% "
          f"{sub['base'].nunique():>13,}  {note}")

# ── Shape ───────────────────────────────────────────────────────────────────
print(f"\n  SHAPE OF THE DRIFT (candidates with shift_max >= {PURE_TREND})")
print(f"    Read from the three period medians. rising/falling is monotone;")
print(f"    humped/dipped means the centre reversed, which a late-minus-early")
print(f"    test would miss entirely -- this is the tga case.")
hi = cand[cand['shift_max'] >= PURE_TREND]
if len(hi):
    for shp, n in hi['shape'].value_counts().items():
        print(f"    {shp:<10} {n:>5}  ({n / len(hi) * 100:4.1f}%)")

# ── Top base factors ────────────────────────────────────────────────────────
print(f"\n  TOP 40 BASE FACTORS BY SHIFT_MAX (candidates)")
print(f"    The three medians show HOW the centre moved.")
bb = (cand.groupby(['table', 'base'])
      .agg(shift=('shift_max', 'max'), shape=('shape', 'first'),
           n=('column', 'count'), e=('med_early', 'first'),
           m=('med_mid', 'first'), l=('med_late', 'first'))
      .reset_index().sort_values('shift', ascending=False))

print(f"\n  {'base factor':<32} {'tbl':<4} {'shift':>7} {'shape':<8} {'n':>3} "
      f"{'med_early':>12} {'med_mid':>12} {'med_late':>12}")
print("  " + "-" * 98)
for _, r in bb.head(40).iterrows():
    print(f"  {str(r['base'])[:31]:<32} {SHORT.get(r['table'], '?'):<4} "
          f"{r['shift']:>7.2f} {r['shape']:<8} {int(r['n']):>3} "
          f"{r['e']:>12.4g} {r['m']:>12.4g} {r['l']:>12.4g}")

# ── By table ────────────────────────────────────────────────────────────────
print(f"\n  BY TABLE (candidates)")
print(f"    {'table':<26} {'cols':>6} {'median':>8} {'>=1.8':>7} {'>=2.5':>7}")
print(f"    {'-' * 58}")
for t in TABLES:
    sub = cand[cand['table'] == t]
    if not len(sub):
        continue
    print(f"    {t:<26} {len(sub):>6} {sub['shift_max'].median():>8.2f} "
          f"{int((sub['shift_max'] >= 1.8).sum()):>7} "
          f"{int((sub['shift_max'] >= 2.5).sum()):>7}")

# ── Already differenced, still drifting ─────────────────────────────────────
dd = sc[(sc['group'] == 'differenced') & (sc['shift_max'] >= 2.0)]
if len(dd):
    print(f"\n  ALREADY DIFFERENCED, STILL DRIFTING: "
          f"{dd['base'].nunique()} base factors at shift_max >= 2.0")
    print(f"    The VOLATILITY of the changes is drifting, not the level.")
    print(f"    Differencing again is wrong. cpi_core_yoy and case_shiller_yoy")
    print(f"    appearing here is inflation genuinely regime-shifting, which is")
    print(f"    signal, not an artefact.")
    top = (dd.groupby('base')['shift_max'].max()
           .sort_values(ascending=False).head(12))
    for b, v in top.items():
        print(f"      {str(b)[:44]:<46} {v:>6.2f}")

# ── Scale-invariant sanity check ────────────────────────────────────────────
# cwskew/cwkurt divide out any rescaling of x, so they SHOULD score low. If they
# do not, either the shape of the cross-section is genuinely changing over time
# (real, and not fixable by transforming the level) or split_moment mislabelled
# something.
si = sc[sc['group'] == 'scale_invariant']
if len(si):
    n_hi = int((si['shift_max'] >= PURE_TREND).sum())
    print(f"\n  SCALE-INVARIANT MOMENTS (cwskew, cwkurt): {len(si)} columns")
    print(f"    median shift_max {si['shift_max'].median():.2f}, "
          f"{n_hi} at >= {PURE_TREND}")
    print(f"    These divide out any rescaling of x, so a low median confirms")
    print(f"    the statistic is measuring level drift rather than something")
    print(f"    every column shares.")

# ── Targeted checks ─────────────────────────────────────────────────────────
print(f"\n  CARRIED-OVER CANDIDATES (from the boundary diagnostic)")
for b in ['csize_to_shrout', 'oi_wt_theta', 'total_n_trades_a_pct',
          'vix_fut_volume', 'tga_diff']:
    g = sc[sc['base'] == b]
    if len(g):
        r = g.loc[g['shift_max'].idxmax()]
        print(f"    {b:<24} shift {r['shift_max']:>6.2f}  {r['shape']:<8} "
              f"{len(g)} col(s)  [{r['group']}]")
    else:
        print(f"    {b:<24} not found")

print(f"\n  CFTC: RAW COUNTS vs OI-NORMALISED")
print(f"    Total VIX futures open interest grew ~10x over the sample, so every")
print(f"    raw count grew with it. If the raw counts drift and the _pct")
print(f"    versions do not, the raw counts are REDUNDANT rather than in need")
print(f"    of a transform -- that is a drop, not a transform.")
for lab, cols in (('raw counts', ['lev_long', 'lev_short', 'am_long',
                                  'am_short', 'dealer_long', 'dealer_short',
                                  'other_long', 'other_short']),
                  ('OI-normalised', ['lev_net_pct', 'am_net_pct',
                                     'dealer_net_pct'])):
    g = sc[sc['base'].isin(cols)]
    if len(g):
        print(f"    {lab:<16} n={len(g):<3} "
              f"median {g['shift_max'].median():>6.2f}  "
              f"max {g['shift_max'].max():>6.2f}")

print(f"\n  Saved: stationarity_scan.csv ({len(sc):,} rows)")
print(f"  Nothing has been transformed. Set the threshold from the table")
print(f"  above, then fill in the transform dict in 03b.")

STAGE 3 CLEANING — 03a: STATIONARITY SCAN

  shift_max = (max - min of period medians) / sigma_f
  A pure noiseless linear trend scores 1.80
  agg_daily_means             5,234 rows x   336 features
  agg_daily_full_moments      5,234 rows x  1016 features
  agg_monthly_means             248 rows x   320 features
  agg_monthly_full_moments      248 rows x  1052 features
  weekly                      3,107 rows x    34 features

  COVERAGE
    Every column is measured. The report segments them because a high
    score means something different in each group.

    group               columns  base factors
    -----------------------------------------
    candidate             1,505           478
    differenced             545           212
    scale_invariant         708           354
    -----------------------------------------
    total                 2,758           690

  SHIFT_MAX DISTRIBUTION — ALL THREE GROUPS
    Percentages are within-group, so the columns are comparable even